# Parsing helper trials

In [1]:
from ireland_property_analysis.utils import get_root_dir

ROOT_DIR = get_root_dir()
print(ROOT_DIR)

/workspaces/ireland-property-analysis


## Prices, currancy.. etc

In [2]:
import pandas as pd

from ireland_property_analysis.utils import get_file_encoding

csv_path = ROOT_DIR / "sample-data" / "PPR-ALL.csv"

sales_data = pd.read_csv(
    csv_path,
    encoding=get_file_encoding(str(csv_path))
)

/tmp/ipykernel_1890/843690001.py:7: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_data = pd.read_csv(


In [3]:
sales_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 767776 entries, 0 to 767775
Data columns (total 9 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   Date of Sale (dd/mm/yyyy)  767776 non-null  object
 1   Address                    767776 non-null  object
 2   County                     767776 non-null  object
 3   Eircode                    220268 non-null  object
 4   Price (Ć)                  767776 non-null  object
 5   Not Full Market Price      767776 non-null  object
 6   VAT Exclusive              767776 non-null  object
 7   Description of Property    767776 non-null  object
 8   Property Size Description  52838 non-null   object
dtypes: object(9)
memory usage: 52.7+ MB


In [4]:
sales_data.loc[1000:1010, ["Price (Ć)"]]

,Price (Ć)
1000,"Ć120,000.00"
1001,"Ć76,666.66"
1002,"Ć10,000.00"
1003,"Ć158,590.23"
1004,"Ć70,000.00"
1005,"Ć6,000.00"
1006,"Ć365,110.13"
1007,"Ć175,000.00"
1008,"Ć130,000.00"
1009,"Ć210,000.00"


In [5]:
from ireland_property_analysis.utils import parse_currency, adjust_price_for_vat

price = parse_currency("Ć190,000.00")
adjusted_price = adjust_price_for_vat(price, vat_exclusive=True)  # Assuming the price is VAT exclusive
print(adjusted_price)  # Output: 215650.00

215650.00000


In [6]:
sales_data.loc[1000:1010, ["Price (Ć)","VAT Exclusive","Not Full Market Price"]]

,Price (Ć),VAT Exclusive,Not Full Market Price
1000,"Ć120,000.00",No,Yes
1001,"Ć76,666.66",No,No
1002,"Ć10,000.00",No,No
1003,"Ć158,590.23",Yes,No
1004,"Ć70,000.00",No,No
1005,"Ć6,000.00",No,No
1006,"Ć365,110.13",Yes,No
1007,"Ć175,000.00",No,No
1008,"Ć130,000.00",No,No
1009,"Ć210,000.00",Yes,No


In [7]:
sales_data.loc[1000:1010, ["Price (Ć)","Property Size Description"]]

,Price (Ć),Property Size Description
1000,"Ć120,000.00",NaN
1001,"Ć76,666.66",NaN
1002,"Ć10,000.00",NaN
1003,"Ć158,590.23",greater than or equal to 38 sq metres and less...
1004,"Ć70,000.00",NaN
1005,"Ć6,000.00",NaN
1006,"Ć365,110.13",greater than or equal to 38 sq metres and less...
1007,"Ć175,000.00",NaN
1008,"Ć130,000.00",NaN
1009,"Ć210,000.00",greater than or equal to 38 sq metres and less...


In [10]:
size_descriptions_without_numbers = (
    sales_data["Property Size Description"]
    .dropna()
    .astype("string")
    .str.replace(r"\d+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip(" -/, ")
)

sorted(size_descriptions_without_numbers.unique())

['greater than or equal to sq metres',
 'greater than or equal to sq metres and less than sq metres',
 'greater than sq metres',
 'less than sq metres',
 'n?os l? n? m?adar cearnach',
 'nĒos m¾ nß n¾ cothrom le mķadar cearnach agus nĒos l· nß mķadar cearnach']

In [4]:

import re


greater_str_example = "greater than 1000 sq meters"
less_str_example = "less than 1000 sq meters"
invalid_str_example = "n?os l? n? m?adar cearnach"
invalid_other_str_example = "nĒos m¾ nß n¾ cothrom le mķadar cearnach agus nĒos l· nß mķadar cearnach"

result = re.findall(r"[0-9]+\.?[0-9]*", invalid_other_str_example)
print(result)  # Output: ['1000.5']

[]


In [5]:
from ireland_property_analysis.utils import parse_sqm
size = parse_sqm(invalid_other_str_example)
print(size)  # Output: {'operator': 'greater than', 'value': 1000.0, 'unit': 'sq meters'}

0
